In [18]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("data/chase_banking.pdf")

documents=loader.load()

print(documents)

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-03T09:29:06-04:00', 'author': 'JPMorgan Chase Bank', 'keywords': 'Chase; total; checking; guide to your account; ada; (PDF)', 'moddate': '2024-10-07T09:59:35-04:00', 'subject': 'Chase Total Checking - A Guide To Your Account', 'title': 'Chase Total Checking - A Guide To Your Account (PDF)', 'trapped': '/Unknown', 'source': 'data/chase_banking.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT\n1\nCHASE TOTAL CHECKING\n®\nA GUIDE TO YOUR ACCOUNT †\nIt’s important that you understand how your Chase Total Checking account works. \nWe’ve created this Guide to explain the fees and some key terms of your personal account.\nMONTHLY \nSERVICE FEE*\nMonthly Service Fee $12\nWays to Avoid the \nMonthly Service Fee\n$0 Monthl

In [48]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    
    separators="",
    chunk_size=1536,
    chunk_overlap=20
)

In [49]:
chunks=[]

for doc in documents:
    
    texts=text_splitter.split_text(doc.page_content)
    chunks.extend(texts)

In [55]:
for chunk in enumerate(chunks):
    print(chunk)

(0, 'HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT\n1\nCHASE TOTAL CHECKING\n®\nA GUIDE TO YOUR ACCOUNT †\nIt’s important that you understand how your Chase Total Checking account works. \nWe’ve created this Guide to explain the fees and some key terms of your personal account.\nMONTHLY \nSERVICE FEE*\nMonthly Service Fee $12\nWays to Avoid the \nMonthly Service Fee\n$0 Monthly Service Fee when you have any ONE  of the following during each \nmonthly statement period:\n•  Electronic deposits made into this account totaling $500 or more, such as \npayments from payroll providers or government benefit providers, by using \n(i) the ACH network, (ii) the Real Time Payment or FedNow SM network, or  \n(ii\ni) third party services that facilitate payments to your debit card using the \nVisa® or Mastercard® network\n• OR, a balance at the beginning of each day of $1,500 or more in this account\n•  OR, an average

In [37]:
embedding_name="sentence-transformers/all-mpnet-base-v2"

In [38]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name=embedding_name)

#### 1.  ChromaDB

In [ ]:
import chromadb
from chromadb.config import Settings


client = chromadb.Client(Settings(chroma_db_impl="duckdb+parquet",
                                    persist_directory="db/"
                                ))

In [6]:
from langchain.vectorstores import Chroma

chroma_vectordb=Chroma.from_documents(documents=documents,
                                      embedding=embedding_model,
                                      persist_directory="chroma_db")

In [13]:
response=chroma_vectordb.similarity_search("how to avoid overdradt", k=3)

print(response)

[Document(metadata={'author': 'JPMorgan Chase Bank', 'creationdate': '2024-10-03T09:29:06-04:00', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'keywords': 'Chase; total; checking; guide to your account; ada; (PDF)', 'moddate': '2024-10-07T09:59:35-04:00', 'page': 1, 'page_label': '2', 'producer': 'Adobe PDF Library 17.0', 'source': 'data/chase_banking.pdf', 'subject': 'Chase Total Checking - A Guide To Your Account', 'title': 'Chase Total Checking - A Guide To Your Account (PDF)', 'total_pages': 4, 'trapped': '/Unknown'}, page_content='HAVE QUESTIONS?  CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT\nCHASE TOTAL CHECKING\n®\n2\nCHASE DEBIT \nC\nARD COVERAGE SM \nA\nND FEES  3\n(Please visit \nw\nww.chase.com/checking/\ndebit-card-coverage  \nf\nor additional details.)\nChase Debit Card Coverage: You can choose how we treat your everyday (not recurring) debit card \ntransactions when you don’t have enough money available.

docker run qdrant/qdrant

docker run -p 6333:6333 qdrant/qdrant

In [20]:
from langchain.vectorstores import Qdrant

qdb_url="http://localhost:6333"

qdrantDB=Qdrant.from_documents(
    
    documents,
    embedding_model,
    url=qdb_url,
    collection_name="test"
    
)

In [77]:
# Create Qdrant Client
from qdrant_client import QdrantClient
from qdrant_client.http import models

client = QdrantClient(
    url="http://localhost:6333", 
    
)

client.recreate_collection(
    collection_name="openAI-collection",
    vectors_config=models.VectorParams(
      size=1024,
      distance=models.Distance.COSINE
    )
)

/var/folders/48/pxhsm41955d2w1g4cmh2g57h0000gn/T/ipykernel_65301/2146763248.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [78]:
import uuid
from openai import OpenAI
from qdrant_client.http.models import PointStruct


openai_client = OpenAI(
  base_url='http://localhost:11434/v1',
  api_key='ollama',
)

points = []
for idx,chunk in enumerate(chunks):
    response = openai_client.embeddings.create(
    input=chunk,
    model="mxbai-embed-large" #OpenAI recommended
    )
    print(response.data[0].embedding)
    embeddings = response.data[0].embedding
    point_id = str(uuid.uuid4())  # Generate a unique ID for the point
    points.append(PointStruct(id=point_id,payload={"text": chunk},vector=embeddings))

client.upsert(
    collection_name="openAI-collection",
    wait=True,
    points=points
)

[-0.01443932, 0.052281488, 0.0027609707, -0.0027296247, 0.0054880627, 0.018850945, -0.0012572196, 0.0037695148, 0.01587563, 0.042874936, 0.054353833, 0.001292094, -0.012381447, 0.010942027, -0.06809214, 0.008164364, 0.0017063305, 0.008546043, -0.007911068, -0.022094958, -0.054551832, -0.009987304, -0.023288462, -0.03501094, -0.0441303, -0.0031044357, 0.010465731, -0.014754522, 0.08700246, 0.0363697, -0.04832836, -0.010203528, -0.013014012, 0.012451515, -0.0106720505, 0.032337807, 0.009096712, -0.031633813, -0.04585019, -0.014783338, -0.014097465, 0.018348347, 0.05601053, 0.026331747, -0.053592198, -0.008440822, -0.029441608, -0.025036095, 0.013485424, -0.017051153, 0.037632447, 0.051629037, 0.021101467, -0.00028775405, -0.0016925284, 0.016487181, -0.04047137, -0.046407204, -0.00019245764, 0.017469108, -0.01493422, -0.020517752, 0.037238486, -0.045586247, -0.0077169593, 0.038815506, -0.034014385, 0.026936572, 0.049811028, -0.033016484, 0.011667521, 0.017059427, -0.04326853, 0.01111474, 

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [62]:
import uuid
from qdrant_client.http.models import PointStruct

points = []
#embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
for idx,chunk in enumerate(chunks):
    
    embeddings = embedding_model.embed_query(chunk)
    print(chunk)
    print(embeddings)
    point_id = str(uuid.uuid4())  # Generate a unique ID for the point
    points.append(PointStruct(id=point_id,payload={"text": chunk},vector=embeddings))

client.upsert(
    collection_name="myCollections",
    wait=True,
    points=points
)

HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT
1
CHASE TOTAL CHECKING
®
A GUIDE TO YOUR ACCOUNT †
It’s important that you understand how your Chase Total Checking account works. 
We’ve created this Guide to explain the fees and some key terms of your personal account.
MONTHLY 
SERVICE FEE*
Monthly Service Fee $12
Ways to Avoid the 
Monthly Service Fee
$0 Monthly Service Fee when you have any ONE  of the following during each 
monthly statement period:
•  Electronic deposits made into this account totaling $500 or more, such as 
payments from payroll providers or government benefit providers, by using 
(i) the ACH network, (ii) the Real Time Payment or FedNow SM network, or  
(ii
i) third party services that facilitate payments to your debit card using the 
Visa® or Mastercard® network
• OR, a balance at the beginning of each day of $1,500 or more in this account
•  OR, an average beginning day balance of 

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [34]:

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [35]:
embeddings = embed_model.get_text_embedding("Hello World!")
print(len(embeddings))
print(embeddings[:5])

384
[-0.003275721101090312, -0.01169079914689064, 0.041559189558029175, -0.03814811259508133, 0.024183064699172974]
